In [1]:
user_id = "user_123"
doc_ids = ["medical_doc_001"]

In [2]:
query = "What is my haemoglobin level?"

In [3]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
  model="text-embedding-medembed-large-v0.1",
  base_url="http://127.0.0.1:1234/v1",
  api_key="what-so-ever",
  check_embedding_ctx_length=False
)

In [4]:
import requests

bioMedClip_api_url = "http://localhost:8001/v1/embeddings/text"

In [5]:
text_embedding = embedding_model.embed_query(query)
result = requests.post(
  bioMedClip_api_url,
  json={"text": query},
  timeout=180
)
img_embedding = result.json()["embedding"]

In [6]:
len(text_embedding), len(img_embedding)

(1024, 512)

In [7]:
from qdrant_client import models

COLLECTION_NAME = "medical_documents"

# Only fields that you intend to filter on
PAYLOAD_INDEXES = {
  "user_id": models.PayloadSchemaType.KEYWORD,
  "doc_id": models.PayloadSchemaType.KEYWORD,
  "element_id": models.PayloadSchemaType.KEYWORD,
  "modality": models.PayloadSchemaType.KEYWORD,
  "ingested_at": models.PayloadSchemaType.DATETIME,
}

def initialize_payload_indexes(
  client,
  collection_name=COLLECTION_NAME
):
  # Ensure collection exists
  if not client.collection_exists(collection_name):
    raise ValueError(
      f"Collection '{collection_name}' does not exist. "
      "Create it with your named-vector configuration first."
    )

  # Get existing indexes
  collection_info = client.get_collection(collection_name)
  existing_indexes = collection_info.payload_schema or {}

  # Create only missing indexes
  for field, schema in PAYLOAD_INDEXES.items():
    if field in existing_indexes:
      print(f"Already indexed: {field}")
      continue

    client.create_payload_index(
      collection_name=collection_name,
      field_name=field,
      field_schema=schema,
      wait=True
    )

    print(f"Created index: {field}")

  print("Payload index initialization complete!")

In [10]:
initialize_payload_indexes(
  client=qdrant_client,)

Created index: user_id
Created index: doc_id
Created index: element_id
Created index: modality
Created index: ingested_at
Payload index initialization complete!


In [8]:
from datetime import datetime, timezone
from qdrant_client import models

def hybrid_multimodal_search(
  client,
  collection_name: str,
  text_embedding: list[float],
  biomedclip_text_embedding: list[float],
  user_id: str,
  doc_ids: list[str] | None = None,
  # content_types: list[str] | None = None,
  page_number: int | None = None,
  ingestion_from: datetime | None = None,
  ingestion_to: datetime | None = None,
  top_k: int = 5,
  prefetch_limit: int = 30,
):
  if len(text_embedding) != 1024:
    raise ValueError("Expected 1024D text embedding")

  if len(biomedclip_text_embedding) != 512:
    raise ValueError("Expected 512D BioMedCLIP embedding")

  # Mandatory user filter
  conditions = [
    models.FieldCondition(
      key="user_id",
      match=models.MatchValue(value=user_id)
    )
  ]

  # Filter by multiple document IDs
  if doc_ids:
    conditions.append(
      models.FieldCondition(
        key="doc_id",
        match=models.MatchAny(any=doc_ids)
      )
    )

  # Filter by content type
  # if content_types:
  #   conditions.append(
  #     models.FieldCondition(
  #       key="content_type",
  #       match=models.MatchAny(any=content_types)
  #     )
  #   )

  # Filter by page number
  # if page_number is not None:
  #   conditions.append(
  #     models.FieldCondition(
  #       key="page_number",
  #       match=models.MatchValue(value=page_number)
  #     )
  #   )

  # Ingestion date range
  if ingestion_from or ingestion_to:
    date_range = models.DatetimeRange(
      gte=ingestion_from,
      lte=ingestion_to
    )

    conditions.append(
      models.FieldCondition(
        key="ingested_at",
        range=date_range
      )
    )

  query_filter = models.Filter(must=conditions)

  # Independent searches
  prefetch = [
    models.Prefetch(
      query=text_embedding,
      using="text_vector",
      filter=query_filter,
      limit=prefetch_limit
    ),
    models.Prefetch(
      query=biomedclip_text_embedding,
      using="image_vector",
      filter=query_filter,
      limit=prefetch_limit
    )
  ]

  # Server-side Reciprocal Rank Fusion
  results = client.query_points(
    collection_name=collection_name,
    prefetch=prefetch,
    query=models.FusionQuery(
      fusion=models.Fusion.RRF
    ),
    limit=top_k,
    with_payload=True,
    with_vectors=False
  )

  return results.points


In [9]:
from qdrant_client import QdrantClient
from dotenv import load_dotenv
import os

load_dotenv()

qdrant_client = QdrantClient(
  url=os.environ.get("QDRANT_URL"),
  api_key=os.environ.get("QDRANT_API_KEY")
)

results = hybrid_multimodal_search(
  client=qdrant_client,
  collection_name="medical_documents",
  text_embedding=text_embedding,
  biomedclip_text_embedding=img_embedding,
  user_id=user_id,
  doc_ids=doc_ids,
  top_k=5,
  prefetch_limit=30
)

In [10]:
results

[ScoredPoint(id='10d56b9f-b976-536a-8c5d-af4bf751f02e', version=7, score=0.5, payload={'user_id': 'user_123', 'doc_id': 'medical_doc_001', 'element_id': '624ae81ef831d57ed30c298d8734bad1_1', 'content_type': 'table', 'modality': 'text', 'source_type': 'unstructured', 'file_name': 'Report.pdf', 'mime_type': 'application/pdf', 'page_number': 1, 'coordinates': {'points': [[141.03102111816406, 621.6292114257812], [141.03102111816406, 2622.771484375], [2442.67333984375, 2622.771484375], [2442.67333984375, 621.6292114257812]], 'system': 'PixelSpace', 'layout_width': 2897, 'layout_height': 4093}, 'text_as_html': '<table><tr><td colspan="4">CBC Haemogram</td></tr><tr><td colspan="4">Investigation Observed Value Unit Biological Reference</td></tr><tr><td colspan="4">Erythrocytes</td></tr><tr><td>Haemoglobin (Hb)</td><td>16.2</td><td>gm/dL</td><td>14-18</td></tr><tr><td>Erythrocyte (RBC) Count</td><td>5.63</td><td>mill/cu.mm</td><td>4.7-6.0</td></tr><tr><td>PCV (Packed Cell Volume)</td><td>48.0</

In [11]:
results[0].payload

{'user_id': 'user_123',
 'doc_id': 'medical_doc_001',
 'element_id': '624ae81ef831d57ed30c298d8734bad1_1',
 'content_type': 'table',
 'modality': 'text',
 'source_type': 'unstructured',
 'file_name': 'Report.pdf',
 'mime_type': 'application/pdf',
 'page_number': 1,
 'coordinates': {'points': [[141.03102111816406, 621.6292114257812],
   [141.03102111816406, 2622.771484375],
   [2442.67333984375, 2622.771484375],
   [2442.67333984375, 621.6292114257812]],
  'system': 'PixelSpace',
  'layout_width': 2897,
  'layout_height': 4093},
 'text_as_html': '<table><tr><td colspan="4">CBC Haemogram</td></tr><tr><td colspan="4">Investigation Observed Value Unit Biological Reference</td></tr><tr><td colspan="4">Erythrocytes</td></tr><tr><td>Haemoglobin (Hb)</td><td>16.2</td><td>gm/dL</td><td>14-18</td></tr><tr><td>Erythrocyte (RBC) Count</td><td>5.63</td><td>mill/cu.mm</td><td>4.7-6.0</td></tr><tr><td>PCV (Packed Cell Volume)</td><td>48.0</td><td>%</td><td>42-52</td></tr><tr><td>MCV (Mean Corpuscular

In [12]:
#12
import sys
from pathlib import Path

# Reuse the backend's async session + models (same pattern as the e2e tests)
sys.path.insert(0, str(Path.cwd() / "Backend"))

from app.database import SessionLocal
from app.models import DocumentImage

In [13]:
# 13
def select_context_points(
  points,
  max_images: int = 2,
):
  """
  One pass over the retrieved points:

  - Images (content_type == "image"): keep only the top `max_images`.
    Points arrive sorted by RRF score, so first-seen = best.
  - Everything else (text / tables): keep all of them.
  """
  ret_points = []   # all text / table points
  # top `max_images` image points
  citations = []
  images_to_fetch = []  ## Images table is indexed by doc_id + image_id, so we can fetch them in one go with this separate list

  for index, point in enumerate(points):
    if point.payload.get("content_type") == "image":
      if len(images_to_fetch) < max_images:
        ret_points.append({
          "evidence_id": "E" + str(index+1),
          "type": "image",
          "image_id": point.payload.get("image_id"),
        })
        images_to_fetch.append((point.payload.get("doc_id"), point.payload.get("image_id")))
    else:
      if point.payload.get("content_type") == "table":
        ret_points.append({
          "evidence_id": "E" + str(index+1),
          "type": "table",
          "content": point.payload.get("text_as_html"),
        })
      else:
        ret_points.append({
          "evidence_id": "E" + str(index+1),
          "type": "text",
          "content": point.payload.get("text"),
        })

    citations.append({
      "id": "E" + str(index+1),
      "user_id": point.payload.get("user_id"),
      "file_name": point.payload.get("file_name"),
      "doc_id": point.payload.get("doc_id"),
      "content_type": point.payload.get("content_type"),
      "page_number": point.payload.get("page_number"),
      "coordinates": point.payload.get("coordinates"),
      "score": point.score
    })

  return ret_points, citations, images_to_fetch

In [14]:
ret_points, citations, images_to_fetch = select_context_points(points=results, max_images=2)

In [15]:
citations

[{'id': 'E1',
  'user_id': 'user_123',
  'file_name': 'Report.pdf',
  'doc_id': 'medical_doc_001',
  'content_type': 'table',
  'page_number': 1,
  'coordinates': {'points': [[141.03102111816406, 621.6292114257812],
    [141.03102111816406, 2622.771484375],
    [2442.67333984375, 2622.771484375],
    [2442.67333984375, 621.6292114257812]],
   'system': 'PixelSpace',
   'layout_width': 2897,
   'layout_height': 4093},
  'score': 0.5},
 {'id': 'E2',
  'user_id': 'user_123',
  'file_name': 'Report.pdf',
  'doc_id': 'medical_doc_001',
  'content_type': 'table',
  'page_number': 1,
  'coordinates': {'points': [[141.03102111816406, 621.6292114257812],
    [141.03102111816406, 2622.771484375],
    [2442.67333984375, 2622.771484375],
    [2442.67333984375, 621.6292114257812]],
   'system': 'PixelSpace',
   'layout_width': 2897,
   'layout_height': 4093},
  'score': 0.33333334},
 {'id': 'E3',
  'user_id': 'user_123',
  'file_name': None,
  'doc_id': 'medical_doc_001',
  'content_type': 'text',


In [16]:
## Next retrieve the stored JPEG bytes from the database using the indexed doc_id and image_id from the images_to_fetch list, then base64-encode them

In [ ]:
from types import SimpleNamespace

import base64
from sqlalchemy import select, tuple_

async def fetch_images_base64(
  images_to_fetch: list[tuple[str, str]],
):
  """
  Retrieve the selected images from the `images` table in a SINGLE DB call
  and base64-encode their stored JPEG bytes for the LLM.

  `image_id` is unique only per document (composite PK = (document_id,
  image_id)), so the query matches the (doc_id, image_id) pairs taken from
  `images_to_fetch`. The user filter was already enforced by Qdrant at
  retrieval time.

  Returns one entry per input, in the SAME order, so the result can be
  zipped straight onto `image_points` by position. An image whose row is
  missing from the DB (e.g. deleted) comes back with image_base64=None so
  the alignment is preserved.
  """
  if not images_to_fetch:
    return []

  stmt = (
    select(
      DocumentImage.document_id,
      DocumentImage.image_id,
      DocumentImage.image_data,
    )
    .where(
      tuple_(DocumentImage.document_id, DocumentImage.image_id).in_(images_to_fetch)
    )
  )

  async with SessionLocal() as db:
    rows = (await db.execute(stmt)).all()

  base64_by_key = {
    (row.document_id, row.image_id): base64.b64encode(row.image_data).decode("ascii")
    for row in rows
  }

  return [
    SimpleNamespace(
      image_id=image_id,
      image_base64=base64_by_key.get((doc_id, image_id)),
    )
    for doc_id, image_id in images_to_fetch
  ]

0.5

In [ ]:
# 19 — LLM prompt + input furnishing + Groq call
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# ---------------------------------------------------------------------------
# Fixed response for insufficient evidence (change wording here only)
# ---------------------------------------------------------------------------
INSUFFICIENT_EVIDENCE_RESPONSE = (
    "I could not find sufficient evidence in your documents to answer this question."
)

SYSTEM_PROMPT = f"""
You are a strict, evidence-grounded medical document assistant.

Your task is to answer questions about the user's medical documents using
ONLY the evidence supplied in the current request.

You are a document analysis assistant, NOT an independent medical advisor.
Your responsibility is to accurately report, summarize, and explain what
the supplied documents explicitly contain.

Never introduce independent medical knowledge, assumptions, diagnoses,
causal relationships, or treatment recommendations.

==================================================
1. UNDERSTANDING THE EVIDENCE
==================================================

You will receive multiple evidence items.

Each evidence item contains:
- evidence_id: A unique ID such as E1, E2, E3.
- type: text, table, or image.
- content: Extracted document content.
- Images may additionally be attached separately.

Treat every evidence item as an independent source.

A. TEXT EVIDENCE:
- Use the information explicitly stated in the text.
- You may summarize its findings without changing their meaning.
- Distinguish between observed findings, screening interpretations,
  suspected conditions, and confirmed diagnoses.

B. TABLE EVIDENCE:
- Table content may be supplied as raw HTML.
- Interpret the HTML table by its row and column structure.
- Correctly associate each test name with its corresponding result,
  unit, and biological reference interval.
- Never mix values from different rows.
- Preserve the exact reported numerical values and units.
- A table containing a measurement does not automatically establish
  a diagnosis or clinical interpretation.
- You may describe a value as below or above its supplied reference
  interval when the comparison is unambiguous.
- Do not independently infer a medical condition from that comparison
  unless the supplied evidence explicitly establishes it.

C. IMAGE EVIDENCE:
- Use image information only when the actual image is provided and
  visually accessible in the current request.
- An image_id or image filename alone is NOT evidence of image contents.
- Do not invent text, values, diagnoses, or findings from image metadata.
- If an image is unavailable or unreadable, treat its contents as unknown.

==================================================
2. STRICT EVIDENCE-ONLY GENERATION
==================================================

1. Every factual statement must be supported by the supplied evidence.

2. Never use your internal medical knowledge to fill missing information.

3. Do not independently derive diagnoses, explanations, causes,
   classifications, or medical recommendations.

4. Do not infer causal relationships between separate findings merely
   because they appear together in the same report.

5. You MAY report medical interpretations that are explicitly stated
   in the supplied evidence.

6. When reporting an interpretation, preserve its original meaning
   and certainty.

7. You may combine information from multiple evidence items only
   when the combined statement is directly supported by those items.

8. Do not confuse an observed laboratory value with its clinical
   interpretation.

9. Never introduce medical facts merely to make the answer more
   comprehensive.

10. Do not provide general medical explanations of conditions,
    diseases, tests, or treatments unless explicitly present
    in the supplied evidence.

11. Do not calculate or independently derive medical indices,
    ratios, scores, or numerical interpretations.

12. Do not assume that missing results are normal, negative,
    or indicative of any particular condition.

==================================================
3. PRESERVE MEDICAL UNCERTAINTY
==================================================

This is a particularly important requirement.

You MUST preserve the certainty and limitations of every statement
in the original evidence.

Examples:

- "Consistent with Beta Thalassemia Trait" must remain
  "consistent with Beta Thalassemia Trait".
  It must NOT become "confirmed Beta Thalassemia Trait".

- "Screens toward Beta Thalassemia Trait" must remain
  a screening interpretation.
  It must NOT become a definitive diagnosis.

- "Screening tool and is not diagnostic on its own" must never
  be omitted when discussing the corresponding screening result.

- "Iron supplementation may be considered" must NOT become
  "You need iron supplementation".

- "After clinical correlation" must remain a condition
  attached to the recommendation.

- "Recommended for confirmation" must NOT become
  "Already confirmed".

- "Indicating concurrent iron deficiency" may be reported as
  the document's interpretation, but must not be expanded into
  an independently established cause of another finding.

Never strengthen or modify the certainty of medical statements.

Use expressions such as:
- "The report states..."
- "The report describes..."
- "The laboratory result shows..."
- "The report interprets this finding as..."
- "The report identifies findings consistent with..."
- "The document recommends considering..."

Clearly distinguish what the report observes from what it concludes.

==================================================
4. MEDICAL RECOMMENDATIONS AND TREATMENT
==================================================

1. Never independently recommend medications, supplements,
   treatments, procedures, or medical tests.

2. You may report recommendations explicitly mentioned in the evidence.

3. Preserve all conditions, qualifications, and uncertainties.

4. Never convert a conditional recommendation into a definite instruction.

5. Never tell the user that they need a particular medication or
   treatment unless the supplied evidence explicitly states that
   recommendation in those terms.

6. If the user asks whether they should take a medication, and the
   evidence only contains a conditional recommendation, do not
   independently decide whether they should take it.

7. Report the document's recommendation accurately or abstain
   when the question requires an unsupported personal medical decision.

==================================================
5. STRICT INLINE CITATION REQUIREMENTS
==================================================

Every factual claim MUST have an inline citation.

Use ONLY this exact citation format:

Single evidence:
[[cite:E1]]

Multiple evidence items:
[[cite:E1, E2]]

CITATION PLACEMENT:

1. Place the citation immediately after the exact sentence or
   clause it supports.

2. Citations may appear in the middle of a paragraph, middle of a
   sentence, or at the end of a sentence, wherever the supported
   claim ends.

3. If a sentence contains two independent factual claims supported
   by different evidence, split them into separate sentences and
   cite each separately.

4. Do not put one citation at the end of an entire paragraph
   containing multiple unrelated claims.

5. Multiple evidence IDs may be used together only when all cited
   items genuinely support the same claim.

6. Use the most directly relevant evidence item.

7. Never invent, modify, renumber, or guess evidence IDs.

8. Never use numbered citations such as [1], footnotes,
   bibliography entries, or alternative citation formats.

9. Every factual statement in headings, summaries, bullet points,
   and conclusions must also have citations.

10. Never generate a citation for evidence that merely mentions
    the same medical topic without supporting the actual claim.

==================================================
6. NUMERICAL AND LABORATORY ACCURACY
==================================================

For laboratory measurements and medical values:

1. Preserve the exact numerical values.
2. Preserve the original units.
3. Preserve the corresponding reference intervals.
4. Do not independently round, normalize, or modify values.
5. Never associate a test result with another test's reference range.
6. Do not confuse values from separate reports or dates.
7. Do not independently calculate derived medical interpretations.
8. When mentioning a reference interval, cite the evidence containing
   that interval.
9. Never assume that a value outside a reference interval establishes
   a particular diagnosis.

For example, if the table contains:
Haemoglobin: 10.2 gm/dL
Reference interval: 14-18 gm/dL

You may report these exact values and state that the measured value
is below the supplied reference interval.

However, do not independently determine the underlying cause.

==================================================
7. HANDLING MULTIPLE EVIDENCE ITEMS
==================================================

Different evidence items may contain overlapping information.

For example:
- One table may contain a numerical result.
- Another text chunk may interpret that result.
- A third text chunk may provide a recommendation.

When answering:

1. Prefer the explicit interpretation in the document over
   independently interpreting raw numerical values.

2. Use the table as supporting evidence for numerical values.

3. Use the relevant text evidence for the interpretation.

4. Cite both sources when a statement combines a numerical result
   with its explicitly documented interpretation.

5. Do not treat repeated information in different chunks as
   independent confirmation.

6. Do not combine separate findings into a causal explanation
   unless the evidence explicitly makes that connection.

7. If different evidence items contradict each other, do not
   arbitrarily select one.

==================================================
8. STRICT INSUFFICIENT EVIDENCE POLICY
==================================================

This rule has the highest priority.

Before answering, determine whether the COMPLETE question can
be answered using the supplied evidence.

Return the following exact response if:

- The requested information is missing.
- The evidence is only loosely related to the question.
- The question requires independent medical knowledge.
- The question requires an unsupported diagnosis or causal explanation.
- The question requires a medical interpretation not stated
  in the supplied evidence.
- A required value, unit, or detail is unavailable.
- Important evidence is contradictory or ambiguous.
- The user asks for a personal treatment decision that cannot
  be established from the documents.
- Only part of a multi-part question can be answered.

Do not partially answer a multi-part question.

Do not guess or fill missing information with general medical knowledge.

Do not provide possible explanations as alternatives.

Do not add a disclaimer to an otherwise unsupported answer.

When evidence is insufficient, reply with EXACTLY:

{INSUFFICIENT_EVIDENCE_RESPONSE}

Nothing else.

==================================================
9. ANSWER FORMATTING
==================================================

1. Answer the user's actual question directly.

2. Use concise, professional, easy-to-understand language.

3. Use Markdown headings and bullet points when useful.

4. Avoid unnecessary introductions and repetitive explanations.

5. Do not add independent conclusions in a final summary.

6. Do not repeat medical findings unnecessarily.

7. Keep related findings together when appropriate.

8. Every factual statement must have an immediate inline citation.

9. Never generate a separate sources section.

10. Never add unsupported medical advice at the end of an answer.

==================================================
10. FINAL SELF-VERIFICATION
==================================================

Before returning your answer, silently verify EVERY factual statement.

For each statement, check:

1. Is it explicitly supported by the supplied evidence?
2. Am I introducing independent medical knowledge?
3. Have I inferred a cause or relationship not stated in the report?
4. Have I changed the certainty of the original interpretation?
5. Have I converted a screening result into a confirmed diagnosis?
6. Have I converted a conditional recommendation into definite advice?
7. Are all numerical values and units exact?
8. Does the citation directly support the associated statement?
9. Are all cited evidence IDs valid?
10. Have I answered every required part of the question?

Remove any unsupported statements.

If removing unsupported information means that the complete
question can no longer be answered, return the exact
insufficient evidence response.

EVIDENCE ACCURACY IS MORE IMPORTANT THAN ANSWER COMPLETENESS.

Never sacrifice factual accuracy to generate a more detailed answer.
"""

# ---------------------------------------------------------------------------
# Groq (OpenAI-compatible) client
# ---------------------------------------------------------------------------
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROQ_MODEL = "qwen/qwen3.8-27b"

# groq_client = OpenAI(
#   base_url=GROQ_BASE_URL,
#   api_key=os.environ.get("GROQ_API_KEY"),
# )

# xkiro_client = OpenAI(
#   base_url="https://api.xkiro.com/v1",
#   api_key=""
# )

# crow_client = OpenAI(
#   base_url="https://crowllm.com/v1/chat/completions",
#   api_key=""
# )

google_client = OpenAI(
  base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
  api_key=""
)


def build_multimodal_messages(
  query: str,
  ret_points: list[dict],
  image_rows,
):
  """
  Furnish the LLM input: system prompt + user query + sources.

  ret_points : evidence items from select_context_points (text / table / image,
               each with an evidence_id)
  image_rows : rows returned by fetch_images_base64 (image_id, image_base64)

  The base64 of each image is attached HERE, while the message parts are
  being built, so ret_points never need a separate pass to be mutated.
  """
  base64_by_id = {row.image_id: row.image_base64 for row in image_rows}

  content = [
    {
      "type": "text",
      "text": f"Question: {query}\n\nEvidence items from the user's documents:",
    }
  ]

  for point in ret_points:
    eid = point["evidence_id"]

    if point["type"] == "image":
      content.append({"type": "text", "text": f"\n[{eid}] (image — attached below)"})

      image_base64 = base64_by_id.get(point["image_id"])
      if image_base64:
        content.append({
          "type": "image_url",
          "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"},
        })
      else:
        content.append({
          "type": "text",
          "text": "(image content unavailable — do NOT cite this item as support)",
        })
    else:
      label = "table (raw HTML)" if point["type"] == "table" else "text"
      content.append({
        "type": "text",
        "text": f"\n[{eid}] ({label})\n{point['content']}",
      })

  return [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": content},
  ]


def answer_question(
  query: str,
  ret_points: list[dict],
  image_rows,
):
  """
  Build the messages and call Groq.

  Returns the raw response text: an answer with [[cite:E#]] tags, or
  INSUFFICIENT_EVIDENCE_RESPONSE (exact-match detectable downstream).
  """
  messages = build_multimodal_messages(query, ret_points, image_rows)

  response = google_client.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=messages,
    temperature=0,
    max_tokens=2048,
  )

  return response.choices[0].message.content

# Usage:
# image_rows = await fetch_images_base64(images_to_fetch)
# answer = answer_question(query, ret_points, image_rows)
# print(answer)

In [18]:
query = "Why is my haemoglobin low, and what does my report say is causing it? Do I need iron or is it something else?"

In [19]:
mock_data_ret_points = [{
  'evidence_id': 'E1',
  'type': 'table',
  'content': """<table><tr><td colspan='4'>CBC Haemogram</td></tr><tr><td>Haemoglobin (Hb)</td><td>10.2</td><td>gm/dL</td><td>14-18</td></tr><tr><td>Erythrocyte (RBC) Count</td><td>5.5</td><td>mill/cu.mm</td><td>4.7-6.0</td></tr><tr><td>PCV (Packed Cell Volume)</td><td>32.0</td><td>%</td><td>42-52</td></tr><tr><td>MCV (Mean Corpuscular Volume)</td><td>65.0</td><td>fL</td><td>82-101</td></tr><tr><td>MCH (Mean Corpuscular Hb)</td><td>18.5</td><td>pgs</td><td>27-31</td></tr><tr><td>RDW (Red Cell Distribution Width)</td><td>16.8</td><td>%</td><td>11.5-14.0</td></tr><tr><td>Mentzer Index (MCV/RBC)</td><td>11.8</td><td/><td/></tr></table>"""
}, {
  'evidence_id': 'E2',
  'type': 'text',
  'content': """Interpretation: The patient presents with microcytic anemia (Hb 10.2 gm/dL, MCV 65 fL).
      Mentzer Index is 11.8 (MCV/RBC). As per the Mentzer Index rule, MI < 13 in microcytic anemia
      screens toward Beta Thalassemia Trait rather than Iron Deficiency Anemia (MI > 13).
      The Mentzer Index is a screening tool and is not diagnostic on its own. Correlation with "
      hemoglobin studies (HbA2 by HPLC) and iron profile is recommended for definitive diagnosis."""
},
{
  'evidence_id': 'E3',
  'type': 'table',
  'content': """<table><tr><td colspan='4'>Iron Profile</td></tr><tr><td>Serum Iron</td><td>28</td><td>ug/dL</td><td>60-170</td></tr><tr><td>Ferritin</td><td>8</td><td>ng/mL</td><td>30-400</td></tr><tr><td>TIBC</td><td>420</td><td>ug/dL</td><td>250-450</td></tr><tr><td>Transferrin Saturation</td><td>6.7</td><td>%</td><td>20-50</td></tr></table>"""
}, {
  'evidence_id': 'E4',
  'type': 'text',
  'content': """HbA2 estimation by HPLC: 5.8% (Biological Reference Interval: < 3.5%). Elevated HbA2 with "
      "microcytic red cell indices is consistent with Beta Thalassemia Trait. Iron studies reveal "
      "low ferritin (8 ng/mL) and low transferrin saturation (6.7%), indicating concurrent iron "
      "deficiency. Advice: iron supplementation may be considered for the iron deficiency after "
      "clinical correlation. Genetic counselling and haemoglobin electrophoresis of family members "
      "are recommended for confirmation of the thalassemia trait."""
}, {
  'evidence_id': 'E5',
  'type': 'image',
  'image_id': 'img_001',
}]

In [20]:
class MockImageRow:
  def __init__(self, image_id, image_base64):
    self.image_id = image_id
    self.image_base64 = image_base64

mock_image_rows = [
  MockImageRow(
    image_id='img_001',
    image_base64='...'
  )
]

In [21]:
answer_question(query = query, ret_points=mock_data_ret_points, image_rows=mock_image_rows)

'The report states that the patient presents with microcytic anemia, with a haemoglobin level of 10.2 gm/dL (reference interval: 14-18 gm/dL) and an MCV of 65.0 fL (reference interval: 82-101 fL) [[cite:E1, E2]]. \n\nRegarding what is causing it, the supplied documents indicate the following:\n* **Beta Thalassemia Trait:** Elevated HbA2 (5.8%, with a biological reference interval of < 3.5%) combined with microcytic red cell indices is consistent with Beta Thalassemia Trait [[cite:E4]].\n* **Concurrent Iron Deficiency:** Iron studies reveal a low ferritin level of 8 ng/mL (reference interval: 30-400 ng/mL) and a low transferrin saturation of 6.7% (reference interval: 20-50%), indicating concurrent iron deficiency [[cite:E3, E4]]. \n\nRegarding whether you need iron, the report states that iron supplementation may be considered for the iron deficiency after clinical correlation [[cite:E4]].'